<a href="https://colab.research.google.com/github/starlton/Deep-Learning/blob/main/Week%202/week2_nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — A Neural Network From Scratch

**Goal:** Build a complete neural network on top of the `micrograd` autograd engine and train it to solve XOR — using nothing but the `Value` class we wrote ourselves.

**The architecture is a nesting doll:**

| Class | What it is | Built from |
|---|---|---|
| `Neuron` | weights + bias, computes `tanh(w·x + b)` | `Value` objects |
| `Layer` | a list of Neurons | `Neuron` objects |
| `MLP` | a list of Layers | `Layer` objects |

Every gradient that flows through this network is computed by the `Value` class from the previous notebook. These three new classes add **no new gradient logic** — they just organize Values into a trainable structure.

---

## The micrograd Engine (from the previous notebook)

This is the `Value` class we built and gradient-checked. It supports `+`, `*`, `**`, negation, subtraction, `tanh`, and `relu`, and computes gradients via `backward()`. Everything below rests on it.

In [1]:
import math
class Value:
  def __init__(self, data, _children=()): #grad, children, operation):
    self.data = data
    self.grad = 0.0 ## Condition
    self._children = set(_children)
    self._backward = lambda: None

  def __repr__(self):
    return f"Value(data={self.data})"


  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other) # Fix bug so we can use raw values => Value(1.0) + 1 works.

    out = Value(self.data + other.data, _children=(self, other))
    def _backward():
      self.grad +=  out.grad
      other.grad += out.grad

    out._backward = _backward
    return out

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other) # Fix bug so we can use raw values => Value(1.0) * 1 works.

    out = Value(self.data * other.data, _children=(self, other))

    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad

    out._backward = _backward
    return out



  def backward(self):
    # 1. Topological order all of the children in the graph
    topo = []
    visited = set()
    def build_topo(node):
      if node not in visited:
        visited.add(node)
        for child in node._children:
          build_topo(child)

        topo.append(node)

    build_topo(self)

    # 2. Seed the output gradient
    self.grad = 1.0

    # 3. Apply the chain rule in reverse topological order
    for node in reversed(topo):
      node._backward()



  # DAY 2 Adding functions

  def __pow__(self, other):
      assert isinstance(other, (int, float)), "only supporting int/float powers"

      out = Value(self.data ** other, _children=(self,))

      # Should be power rule for computing gradient
      def _backward():
        self.grad += other * (self.data**(other-1)) * out.grad
        #other.grad += self.data * (other**(self.data-1)) * out.grad  ------------------ NO reason to compute others gradient, normally a fixed val anyways.

      out._backward = _backward
      return out

  def __neg__(self):
    return self * -1

  def __sub__(self, other):
    return self + (-other)

  def __rsub__(self, other):
    return other + (-self)

  def __radd__(self, other):
    return self + other

  def tanh(self):
    out = Value(math.tanh(self.data), _children=(self,))

    def _backward():
      self.grad += (1 - out.data**2) * out.grad

    out._backward = _backward
    return out

  def relu(self):
    out = Value(max(0, self.data), _children=(self,))

    def _backward():
      self.grad += (1 if self.data > 0 else 0) * out.grad

    out._backward = _backward
    return out

---
## 1. The Neuron

A single neuron is the smallest unit. It holds:
- **weights** — one `Value` per input, randomly initialized
- **bias** — one `Value`

Its forward pass computes:

$$\text{output} = \tanh\left(\sum_i w_i x_i + b\right)$$

The weighted sum $w \cdot x + b$ produces a single number, then `tanh` squashes it into the range $(-1, 1)$ — this nonlinearity is what lets networks learn complex patterns.

The `parameters()` method returns every trainable Value (all weights + bias). We'll need it to update the neuron during training.

**Why tanh and not ReLU here?** For a tiny network on XOR, tanh converges far more reliably. ReLU outputs 0 for negative inputs, and with only 4 neurons per layer, several can get stuck at 0 and stop learning ("dead ReLUs"). tanh's smooth curve avoids this.

In [2]:
import random

class Neuron:
  def __init__(self, n_inputs):
    # One weight Value per input, random between -1, 1
    self.w = [Value(random.uniform(-1, 1)) for _ in range(n_inputs)]

    # One bias Value between -1, 1
    self.b = Value(random.uniform(-1, 1))

  def __call__(self, x):
    # multiply each weight by its matching input, sum them all, add bias
    act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
    # Uncomment for tanh activation
    return act.tanh()
    # Uncomment below for ReLU
    #return act.relu()

  def parameters(self):
    return self.w + [self.b]

**Test:** a 2-input neuron should output a single Value in $(-1, 1)$ and report 3 parameters (2 weights + 1 bias).

In [3]:
# Testing Neuron Class
random.seed(12) # seed 12 yields positive number, reliable 123 is negative :(

n = Neuron(2)
x = [Value(1.0), Value(2.0)]
out = n(x)
print(out)
print(len(n.parameters())) # Should be 3, two weights and one bias.

Value(data=0.7220201172414913)
3


---
## 2. The Layer

A layer is simply **several neurons side by side, all receiving the same input**. Feed it an input `x`, and it passes that same `x` to every neuron, collecting each neuron's output into a list.

So a layer of 4 neurons turns one input vector into 4 outputs.

The small `out[0] if len(out) == 1` trick: if a layer has just one neuron (like a final output layer), return a single Value instead of a list of one — cleaner to chain.

Its `parameters()` gathers all parameters from all its neurons into one flat list.

In [4]:
class Layer:
  def __init__(self, n_inputs, n_neurons):
    # Create n_neurons, each expecting n_inputs
    self.neurons = [Neuron(n_inputs) for _ in range(n_neurons)]

  def __call__(self, x):
    # Run x through every neuron, collecting outputs
    out = [n(x) for n in self.neurons]

    # if there's only one neuron, return it directly (not a list of one)
    return out[0] if len(out) == 1 else out

  def parameters(self):
    return [p for neuron in self.neurons for p in neuron.parameters()]

**Test:** a layer with 2 inputs and 4 neurons should give 4 outputs and 12 parameters (4 neurons × 3 params each).

In [5]:
# Testing Layer Class

layer = Layer(2, 4)            # 2 inputs, 4 neurons
x = [Value(1.0), Value(2.0)]
outs = layer(x)
print(len(outs))               # should be 4 (one output per neuron)
print(len(layer.parameters())) # should be 12: 4 neurons × 3 params each

4
12


---
## 3. The MLP (Multi-Layer Perceptron)

The full network: **a stack of layers where each layer's output becomes the next layer's input.** That chaining is what makes it "deep."

### Connecting the layer sizes
For shape `[2, 4, 4, 1]` (2 inputs → hidden 4 → hidden 4 → 1 output), each layer's *input count* must equal the previous layer's *neuron count*:

| Layer | Inputs | Neurons |
|---|---|---|
| 1 | 2 | 4 |
| 2 | 4 | 4 |
| 3 | 4 | 1 |

The line `sizes = [n_inputs] + layer_sizes` builds `[2, 4, 4, 1]`, and the comprehension pairs consecutive sizes: (2,4), (4,4), (4,1).

The forward pass is the whole network in three lines — `x` starts as the input, and each loop iteration replaces it with that layer's output.

In [6]:
class MLP:
  def __init__(self, n_inputs, layer_sizes):
    # full list of sizes: [n_inputs, then each layer size]
    sizes = [n_inputs] + layer_sizes

    # create a layer between each consecutive pair of sizes
    self.layers = [Layer(sizes[i], sizes[i+1]) for i in range(len(layer_sizes))]

  def __call__(self, x):
    # pass x through each layer in sequence
    for layer in self.layers:
      x = layer(x)             # output of this layer becomes input to the next
    return x

  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()]

**Test:** an MLP of shape `[2, 4, 4, 1]` outputs a single Value and has **37 parameters**:
- Layer 1 (2→4): 4 × (2+1) = 12
- Layer 2 (4→4): 4 × (4+1) = 20
- Layer 3 (4→1): 1 × (4+1) = 5
- Total: **37**

In [7]:
# Testing MLP Class

model = MLP(2, [4, 4, 1])
x = [Value(1.0), Value(2.0)]
print(model(x))
print(len(model.parameters()))

Value(data=-0.2537068087832986)
37


In [8]:
for i, layer in enumerate(model.layers):
    print(f"Layer {i}: {len(layer.parameters())} params, {len(layer.neurons)} neurons")

Layer 0: 12 params, 4 neurons
Layer 1: 20 params, 4 neurons
Layer 2: 5 params, 1 neurons


---
## 4. Training on XOR

### Why XOR is the classic test
XOR maps: (0,0)→0, (0,1)→1, (1,0)→1, (1,1)→0. The two output-1 points sit on one diagonal, the two output-0 points on the other. **No single straight line can separate them** — so a single-layer network mathematically cannot solve XOR. It needs a hidden layer to bend the decision boundary. This is the famous problem that nearly killed neural network research in 1969.

### The training loop — four steps per iteration

$$\textbf{1. Forward} \rightarrow \textbf{2. Loss} \rightarrow \textbf{3. Backward} \rightarrow \textbf{4. Update}$$

1. **Forward:** run all 4 XOR inputs through the network → 4 predictions
2. **Loss:** mean squared error, $\sum (\text{pred} - \text{target})^2$ — one number measuring total wrongness. Squaring makes errors positive and punishes large errors more.
3. **Zero grads, then backward:** reset every `p.grad = 0` *before* `loss.backward()`. Because `_backward` uses `+=`, gradients would otherwise accumulate across iterations. **This is the #1 bug source — never skip the zero-grad step.**
4. **Update:** nudge each parameter *opposite* its gradient (downhill on the loss): `p.data -= lr * p.grad`. We subtract because the gradient points uphill and we want to minimize.

In [14]:
# XOR dataset
xs = [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]]
ys = [0.0, 1.0, 1.0, 0.0]

model = MLP(2, [4, 4, 1])

for step in range(200):
    # 1. FORWARD: predict all 4 examples
    ypreds = [model(x) for x in xs]

    # 2. LOSS: mean squared error
    loss = sum((ypred - ygt)**2 for ygt, ypred in zip(ys, ypreds))

    # 3. ZERO GRADS then BACKWARD
    for p in model.parameters():
        p.grad = 0.0          # <-- the critical zero-grad step
    loss.backward()

    # 4. UPDATE: nudge every parameter downhill
    for p in model.parameters():
        p.data -= 0.05 * p.grad

    if step % 10 == 0:
        print(f"step {step}: loss {loss.data}")

step 0: loss 1.089337878877346
step 10: loss 0.9525967189401692
step 20: loss 0.897966222305808
step 30: loss 0.8250623501149981
step 40: loss 0.7216352435275931
step 50: loss 0.5763840178233647
step 60: loss 0.39377508465816974
step 70: loss 0.22437788240368733
step 80: loss 0.120412324521175
step 90: loss 0.0698200765771207
step 100: loss 0.045098693743279475
step 110: loss 0.0318225076853064
step 120: loss 0.02395890337592895
step 130: loss 0.018910762974119776
step 140: loss 0.015459353721679983
step 150: loss 0.013041841828663182
step 160: loss 0.4107275069312906
step 170: loss 0.20385207174401979
step 180: loss 0.10665348300208487
step 190: loss 0.06585378005199544


### Reading the loss curve

The loss starts high (~2.7) and plunges toward near-zero. The steep drop around steps 100–160 is the network suddenly "getting" XOR — bending its boundary to separate the diagonal pairs.

**On late bumps:** if the loss jumps back up near the end, that's the fixed learning rate (0.05) slightly overshooting the minimum — like a stride too big that steps over the bottom of a valley. Fixes: lower the learning rate over time, or stop training once the loss bottoms out.

---
## 5. Verify — Did It Learn XOR?

Predictions should land near the targets: ~0 for (0,0) and (1,1), ~1 for (0,1) and (1,0). The values won't be exactly 0 and 1 (tanh outputs a range), but each should be clearly on the correct side.

In [10]:
for x, y in zip(xs, ys):
    pred = model(x)
    print(f"input {x} -> predicted {pred.data:.3f}, target {y}")

input [0.0, 0.0] -> predicted -0.225, target 0.0
input [0.0, 1.0] -> predicted 0.856, target 1.0
input [1.0, 0.0] -> predicted 0.862, target 1.0
input [1.0, 1.0] -> predicted -0.107, target 0.0


---
## Summary — What We Built

A complete, trainable neural network resting entirely on a hand-built autograd engine:

| Component | Role |
|---|---|
| `Value` | Scalar autograd — computes gradients through any expression |
| `Neuron` | `tanh(w·x + b)` — the smallest learning unit |
| `Layer` | Several neurons sharing an input |
| `MLP` | Stacked layers — the full network |
| Training loop | Forward → loss → backward → update, repeated |

**The achievement:** every one of the 37 gradients in this network was computed by code written from scratch — no PyTorch, no TensorFlow. The network learned XOR, a function that is provably impossible for a single-layer network, by bending its decision boundary through hidden layers.

This is the **Phase 1 checkpoint**: implement gradient descent, build an autograd engine, derive backprop, and train a network from scratch — all complete.

**Next: Probability, statistics, and logistic regression on real data (Titanic).**